In [18]:
import pandas as pd
import numpy as np

In [19]:
QEDS_PATH = "data/processed/WB_QEDS/QEDS_SDDS.csv"
MSCI_PATH = "data/processed/MSCI_indices/mscicountryindex.csv"
CDS_PATH = "data/processed/CDS/Weekly_CDS.csv"
OVX_PATH = "data/processed/Macroeconomic_variables/OVXCLS.csv"

In [20]:
#Debt stats

DEBT_INDICATORS = {
    'debt_st': 'Gross Ext. Debt Pos., General Government, Short-term, All instruments, USD',
    'debt_lt': 'Gross Ext. Debt Pos., General Government, Long-term, All instruments, USD',
    'debt_total': 'Gross Ext. Debt Pos., General Government, All maturities, All instruments, USD',
}

DEFAULT_BARRIER_FORMULA = lambda st, lt: st + 0.5 * lt

def load_qeds(path: str) -> pd.DataFrame:
    """
    Load QEDS data (wide format) and reshape to long format.
    
    Input format:
        Cleaned_Name | Indicator Name | 2000Q1 | 2000Q2 | ...
        Brazil       | Gross Ext...   | 1000   | 1200   | ...
    
    Output format:
        country | indicator | quarter | value
    """
    df = pd.read_csv(path, delimiter=';')
    
    # Identify quarter columns (format: YYYYQn)
    quarter_cols = [c for c in df.columns if c not in ['Cleaned_Name', 'Indicator Name'] 
                    and len(c) == 6 and 'Q' in c]
    
    print(f"  Found {len(quarter_cols)} quarters: {quarter_cols[0]} to {quarter_cols[-1]}")
    
    # Melt to long format
    df_long = df.melt(
        id_vars=['Cleaned_Name', 'Indicator Name'],
        value_vars=quarter_cols,
        var_name='quarter',
        value_name='value'
    )
    
    df_long.columns = ['country', 'indicator', 'quarter', 'value']
    
    # Convert quarter to datetime (end of quarter)
    df_long['date'] = pd.to_datetime(
        df_long['quarter'].str.replace('Q1', '-03-31')
                          .str.replace('Q2', '-06-30')
                          .str.replace('Q3', '-09-30')
                          .str.replace('Q4', '-12-31')
    )
    
    # Convert value to numeric
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')
    
    print(f"  Loaded {len(df_long)} rows, {df_long['country'].nunique()} countries")
    
    return df_long


def extract_debt_variables(qeds_long: pd.DataFrame) -> pd.DataFrame:
    """
    Extract General Government debt variables and pivot to wide by country-date.
    
    Output:
        country | date | debt_st | debt_lt | debt_total | default_barrier
    """    
    # Check which indicators exist
    available = qeds_long['indicator'].unique()
    print(f"  Total unique indicators in data: {len(available)}")
    
    # Find matching indicators
    matched = {}
    for key, pattern in DEBT_INDICATORS.items():
        # Try exact match first
        if pattern in available:
            matched[key] = pattern
        else:
            # Try partial match
            matches = [ind for ind in available if pattern.lower() in ind.lower()]
            if matches:
                matched[key] = matches[0]
                print(f"  WARNING: Using partial match for {key}: {matches[0]}")
    
    print(f"  Matched indicators: {list(matched.keys())}")
    
    if not matched:
        print("\n  ERROR: No debt indicators found!")
        print("  Sample indicators in data:")
        for ind in list(available)[:20]:
            print(f"    {ind}")
        raise ValueError("No matching debt indicators found")
    
    # Filter to matched indicators
    debt_data = qeds_long[qeds_long['indicator'].isin(matched.values())].copy()
    
    # Map indicator names to short names
    inv_matched = {v: k for k, v in matched.items()}
    debt_data['var'] = debt_data['indicator'].map(inv_matched)
    
    # Pivot to wide format
    debt_wide = debt_data.pivot_table(
        index=['country', 'date'],
        columns='var',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Compute default barrier
    if 'debt_st' in debt_wide.columns and 'debt_lt' in debt_wide.columns:
        debt_wide['default_barrier'] = DEFAULT_BARRIER_FORMULA(
            debt_wide['debt_st'].fillna(0),
            debt_wide['debt_lt'].fillna(0)
        )
        print(f"  Computed default barrier using ST + 0.5*LT")
    elif 'debt_st_rem' in debt_wide.columns:
        debt_wide['default_barrier'] = debt_wide['debt_st_rem']
        print(f"  Using ST remaining maturity as default barrier")
    else:
        print("  WARNING: Could not compute default barrier")
    
    print(f"  Output: {len(debt_wide)} country-quarter observations")
    
    return debt_wide
def expand_debt_to_weekly(debt_quarterly: pd.DataFrame, weekly_dates: pd.DataFrame) -> pd.DataFrame:
    """
    Forward-fill quarterly debt to weekly frequency.
    """
    print("Expanding quarterly debt to weekly (forward-fill)...")
    
    debt_weekly_list = []
    
    for country in debt_quarterly['country'].unique():
        # Get quarterly debt for this country
        cq = debt_quarterly[debt_quarterly['country'] == country].copy()
        if len(cq) == 0:
            continue
        
        # Get weeks within debt data range
        min_d, max_d = cq['date'].min(), cq['date'].max()
        weeks = weekly_dates[(weekly_dates['date'] >= min_d) & 
                             (weekly_dates['date'] <= max_d + pd.Timedelta(days=100))]['date'].tolist()
        
        if len(weeks) == 0:
            continue
        
        # Create weekly dataframe and merge quarterly debt
        wdf = pd.DataFrame({'date': weeks, 'country': country})
        wdf = wdf.merge(cq[['date', 'debt_st', 'debt_lt', 'default_barrier']], on='date', how='left')
        
        # Forward fill
        wdf = wdf.sort_values('date')
        wdf[['debt_st', 'debt_lt', 'default_barrier']] = wdf[['debt_st', 'debt_lt', 'default_barrier']].ffill()
        
        debt_weekly_list.append(wdf)
    
    debt_weekly = pd.concat(debt_weekly_list, ignore_index=True)
    print(f"  Output: {len(debt_weekly)} country-week observations")
    
    return debt_weekly

In [21]:
# =============================================================================
# LOAD MSCI
# =============================================================================

def load_msci(path: str) -> pd.DataFrame:
    """
    Load MSCI country indices (daily).
    
    Input format:
        Date       | Brazil | Russia | ...
        01.01.2020 | 100.5  | 200.3  | ...
    
    Output format (long):
        date | country | msci_index
    """
    print(f"Loading MSCI from {path}...")
    df = pd.read_csv(path, delimiter=',')
    
    # Find date column
    date_col = df.columns[0]  # Assume first column is date
    print(f"  Date column: {date_col}")
    
    # Parse date (DD.MM.YYYY)
    df['date'] = pd.to_datetime(df[date_col], format='%d.%m.%Y', errors='coerce')
    
    # If that fails, try other formats
    if df['date'].isna().all():
        df['date'] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    
    # Drop original date column
    df = df.drop(columns=[date_col])
    
    # Get country columns (everything except 'date')
    country_cols = [c for c in df.columns if c != 'date']
    print(f"  Found {len(country_cols)} countries")
    
    # Melt to long format
    df_long = df.melt(
        id_vars=['date'],
        value_vars=country_cols,
        var_name='country',
        value_name='msci_index'
    )
    
    df_long['msci_index'] = pd.to_numeric(df_long['msci_index'], errors='coerce')
    
    # Drop NAs
    df_long = df_long.dropna(subset=['date', 'msci_index'])
    
    print(f"  Date range: {df_long['date'].min()} to {df_long['date'].max()}")
    print(f"  Loaded {len(df_long)} daily observations")
    
    return df_long


def compute_msci_weekly(msci_daily: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate daily MSCI to weekly (Friday end-of-week + volatility).
    """
    print("Computing weekly MSCI statistics...")
    
    df = msci_daily.copy()
    
    # Assign week-ending Friday
    df['week'] = df['date'].apply(
        lambda d: d + pd.Timedelta(days=(4 - d.dayofweek) % 7) if d.dayofweek <= 4 
                  else d + pd.Timedelta(days=(4 - d.dayofweek + 7))
    )
    
    # Compute log returns
    df = df.sort_values(['country', 'date'])
    df['log_ret'] = df.groupby('country')['msci_index'].transform(
        lambda x: np.log(x / x.shift(1))
    )
    
    # Aggregate by country-week
    weekly = df.groupby(['country', 'week']).agg(
        msci_index=('msci_index', 'last'),       # End-of-week level
        msci_ret_weekly=('log_ret', 'sum'),      # Weekly return
        n_obs=('log_ret', 'count')
    ).reset_index()
    
    weekly.columns = ['country', 'date', 'msci_index', 'msci_ret_weekly', 'n_obs']
    
    # Rolling 52-week annualized volatility
    weekly = weekly.sort_values(['country', 'date'])
    weekly['msci_vol_annual'] = weekly.groupby('country')['msci_ret_weekly'].transform(
        lambda x: x.rolling(window=52, min_periods=12).std() * np.sqrt(52)
    )
    
    print(f"  Output: {len(weekly)} country-week observations")
    
    return weekly

In [22]:
# =============================================================================
# MERGE DATASETS
# =============================================================================

def merge_cca_panel(debt: pd.DataFrame, msci: pd.DataFrame) -> pd.DataFrame:
    """
    Merge debt and MSCI data on country-date.
    """
    print("Merging debt and MSCI data...")
    
    # Standardize country names (basic cleaning)
    debt['country_clean'] = debt['country'].str.strip().str.lower()
    msci['country_clean'] = msci['country'].str.strip().str.lower()
    
    # Check overlap
    debt_countries = set(debt['country_clean'].unique())
    msci_countries = set(msci['country_clean'].unique())
    overlap = debt_countries & msci_countries
    
    print(f"  QEDS countries: {len(debt_countries)}")
    print(f"  MSCI countries: {len(msci_countries)}")
    print(f"  Overlap: {len(overlap)}")
    
    if len(overlap) == 0:
        print("\n  WARNING: No country name overlap!")
        print(f"  QEDS sample: {list(debt_countries)[:10]}")
        print(f"  MSCI sample: {list(msci_countries)[:10]}")
    
    # Merge
    panel = pd.merge(
        debt,
        msci,
        on=['country_clean', 'date'],
        how='inner',
        suffixes=('_qeds', '_msci')
    )
    
    # Use QEDS country name as primary
    if 'country_qeds' in panel.columns:
        panel['country'] = panel['country_qeds']
        panel = panel.drop(columns=['country_qeds', 'country_msci', 'country_clean'])
    else:
        panel = panel.drop(columns=['country_clean'])
    
    print(f"  Merged panel: {len(panel)} observations, {panel['country'].nunique()} countries")
    
    return panel

In [23]:
print("="*60)
print("BUILDING CCA PANEL DATASET")
print("="*60 + "\n")

# Load QEDS
qeds_long = load_qeds(QEDS_PATH)
debt = extract_debt_variables(qeds_long)

print()

# Load MSCI
msci_daily = load_msci(MSCI_PATH)
msci_weekly = compute_msci_weekly(msci_daily)
weekly_dates = msci_weekly[['date']].drop_duplicates()

print()

#Get weekly debt values

debt_weekly = expand_debt_to_weekly(debt, weekly_dates)

# Merge
panel = merge_cca_panel(debt_weekly, msci_weekly)

print()

# Summary
print("="*60)
print("PANEL SUMMARY")
print("="*60)
print(f"Date range: {panel['date'].min()} to {panel['date'].max()}")
print(f"Countries: {panel['country'].nunique()}")
print(f"Total observations: {len(panel)}")
print(f"\nColumns: {list(panel.columns)}")
print(f"\nSample:\n{panel.head(10)}")

# Check for missing
print(f"\nMissing values:")
print(panel.isnull().sum())

# Save
#Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
#panel.to_csv(OUTPUT_PATH, index=False)
#print(f"\nSaved to {OUTPUT_PATH}")

BUILDING CCA PANEL DATASET

  Found 110 quarters: 1998Q1 to 2025Q2
  Loaded 25542000 rows, 129 countries
  Total unique indicators in data: 1800
  Matched indicators: ['debt_st', 'debt_lt', 'debt_total']
  Computed default barrier using ST + 0.5*LT
  Output: 8931 country-quarter observations

Loading MSCI from data/processed/MSCI_indices/mscicountryindex.csv...
  Date column: Date
  Found 85 countries
  Date range: 1999-12-31 00:00:00 to 2024-12-31 00:00:00
  Loaded 470553 daily observations
Computing weekly MSCI statistics...


/Users/juanfranciscoperez/commodities_and_sovereigns/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


  Output: 94201 country-week observations

Expanding quarterly debt to weekly (forward-fill)...
  Output: 112142 country-week observations
Merging debt and MSCI data...
  QEDS countries: 129
  MSCI countries: 85
  Overlap: 65
  Merged panel: 61958 observations, 65 countries

PANEL SUMMARY
Date range: 1999-12-31 00:00:00 to 2025-01-03 00:00:00
Countries: 65
Total observations: 61958

Columns: ['date', 'debt_st', 'debt_lt', 'default_barrier', 'msci_index', 'msci_ret_weekly', 'n_obs', 'msci_vol_annual', 'country']

Sample:
        date       debt_st       debt_lt  default_barrier  msci_index  \
0 2010-12-03  5.181000e+09  5.237300e+10     3.136750e+10    1034.304   
1 2010-12-10  5.181000e+09  5.237300e+10     3.136750e+10    1063.641   
2 2010-12-17  5.181000e+09  5.237300e+10     3.136750e+10    1021.676   
3 2010-12-24  5.181000e+09  5.237300e+10     3.136750e+10    1029.518   
4 2010-12-31  7.443373e+09  6.204517e+10     3.846596e+10    1044.573   
5 2011-01-07  7.443373e+09  6.204517

In [24]:
ovx = pd.read_csv(OVX_PATH)
ovx['Date'] = pd.to_datetime(ovx['Date'])
ovx.columns = ['date','OVX']
ovx.set_index('date',inplace=True)
weekly_ovx_df = ovx.resample('W-FRI').ffill().dropna()


In [25]:


panel = panel.merge(weekly_ovx_df, on='date')

panel.to_csv('structural_model_panel.csv')

In [26]:
panel

,date,debt_st,debt_lt,default_barrier,msci_index,msci_ret_weekly,n_obs,msci_vol_annual,country,OVX
0,2010-12-03,5.181000e+09,5.237300e+10,3.136750e+10,1034.304,0.033729,3,NaN,Argentina,31.70
1,2010-12-10,5.181000e+09,5.237300e+10,3.136750e+10,1063.641,0.027969,5,NaN,Argentina,30.36
2,2010-12-17,5.181000e+09,5.237300e+10,3.136750e+10,1021.676,-0.040254,5,NaN,Argentina,26.82
3,2010-12-31,7.443373e+09,6.204517e+10,3.846596e+10,1044.573,0.014517,5,NaN,Argentina,29.48
4,2011-01-07,7.443373e+09,6.204517e+10,3.846596e+10,1049.712,0.004908,5,NaN,Argentina,29.64
...,...,...,...,...,...,...,...,...,...,...
50704,2024-12-06,9.624670e+11,7.771480e+12,4.848207e+12,4011.334,0.009039,5,0.125530,United States,30.98
50705,2024-12-13,9.624670e+11,7.771480e+12,4.848207e+12,3975.270,-0.009031,5,0.124161,United States,34.92
50706,2024-12-20,9.624670e+11,7.771480e+12,4.848207e+12,3884.761,-0.023031,5,0.127125,United States,29.27
50707,2024-12-27,9.624670e+11,7.771480e+12,4.848207e+12,3906.430,0.005562,5,0.127121,United States,30.21
